<h3>Análise Exploratória de Dados</h3>

<p>Na fase de EDA, vamos explorar melhor os dados do dataset e validar ou rejeitar as seguintes hipóteses:</p>
<ol>
    <li><b>Desempenho por Região (Ticket Médio):</b> A região Sudeste apresenta um ticket médio por pedido significativamente maior do que a região Sul, justificando um maior investimento em marketing nessa área?</li>
    <li><b>Análise de Taxas de Cancelamento:</b> A categoria "Móveis" possui a maior taxa de pedidos cancelados ou devolvidos em comparação com "Eletrônicos" e "Informática"?</li>
    <li><b>Taxa de pedidos pendentes ou cancelados:</b> Existe uma concentração maior de vendas com status "Pendente" ou "Cancelado" na segunda quinzena do mês em comparação com a primeira quinzena?</li>
    <li><b>Desempenho de Produtos (Faturamento):</b> O produto "Notebook" é responsável pela maior parcela do faturamento total, superando os demais produtos?</li>
    <li><b>Desempenho dos Vendedores:</b> Os vendedores apresentam diferenças significativas no faturamento médio por venda, indicando que alguns vendedores possuem maior desempenho comercial?</li>
</ol>

In [52]:
import pandas as pd

In [53]:
df = pd.read_excel("../data/tratado/dataset_comercial_tratado.xlsx")
df.head()

,ID_Venda,Data,Vendedor,Regiao,Produto,Categoria,Qtd,Preco_Unit,Total,Status_Pagamento
0,1,2026-01-15,João Silva,Sudeste,Notebook,Informática,2,3500.0,7000.0,Pago
1,2,2026-01-16,Maria,Sul,Monitor,Eletrônicos,1,800.5,800.5,Pendente
2,3,2026-01-18,João Silva,Sudeste,Notebook,Informática,1,3500.0,3500.0,Pago
3,4,2026-01-20,Carlos Souza,Nordeste,Teclado Mecânico,Informática,5,150.0,750.0,Pago
4,5,2026-01-22,Ana Lima,Não informado,Mouse sem fio,Informática,2,80.0,160.0,Cancelado


In [54]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4853 entries, 0 to 4852
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ID_Venda          4853 non-null   int64         
 1   Data              4853 non-null   datetime64[ns]
 2   Vendedor          4853 non-null   object        
 3   Regiao            4853 non-null   object        
 4   Produto           4853 non-null   object        
 5   Categoria         4853 non-null   object        
 6   Qtd               4853 non-null   int64         
 7   Preco_Unit        4853 non-null   float64       
 8   Total             4853 non-null   float64       
 9   Status_Pagamento  4853 non-null   object        
dtypes: datetime64[ns](1), float64(2), int64(2), object(5)
memory usage: 379.3+ KB


In [55]:
df.describe()

,ID_Venda,Data,Qtd,Preco_Unit,Total
count,4853.000000,4853,4853.000000,4853.000000,4853.00000
mean,2495.306202,2025-10-11 13:17:35.594477824,10.289924,2505.404218,26359.69151
min,1.000000,2025-01-01 00:00:00,0.000000,51.190000,0.00000
25%,1240.000000,2025-05-23 00:00:00,5.000000,1219.450000,6948.24000
50%,2494.000000,2025-10-06 00:00:00,10.000000,2471.680000,19952.94000
75%,3750.000000,2026-02-25 00:00:00,15.000000,3695.090000,39713.24000
max,4998.000000,2026-12-06 00:00:00,20.000000,5962.530000,116088.80000
std,1446.627453,NaN,5.838918,1473.467509,23233.79493


<h4>Desempenho por Região (Ticket Médio): </h4>

<h5>Hipótese:</h5>
<h5>A região Sudeste apresenta um ticket médio por pedido significativamente maior do que a região Sul, justificando um maior investimento em marketing nessa área?</h5>

In [56]:
df.groupby('Regiao')['Total'].mean().sort_values(ascending=False).to_frame()

,Total
Regiao,
Sudeste,27066.579255
Não informado,26388.110526
Sul,26228.752222
Norte,26095.611070
Nordeste,1264.291667


<p>Após agrupar a média do valor total das vendas por região, pude constatar que a <b>região sudeste</b>, de fato, possui um <b>ticket médio maior</b> que a <b>região sul</b>. Logo, o investimento em marketing deve ser potencializado nessa região.</p>
<p>Em contraste, devemos olhar para a região nordeste também. Por que o ticket médio dessa região ficou muito abaixo das outras? Qual o tipo de campanha de marketing podemos rodar nessa região para atrair mais clientes? Essas são perguntas que podem nos ajudar a entender a situação e traçar um plano de ação.</p>

<p>Aproveitando que a região sudeste é a região com o maior ticket médio, posso abrir essa região por produto para ver o que mais se vende.</p>

In [57]:
df_sudeste = df[df['Regiao'] == 'Sudeste']
df_sudeste.groupby('Produto')['Total'].mean().sort_values(ascending=False).to_frame()

,Total
Produto,
Mouse,30134.841739
Teclado,28979.691729
Headset,27236.962213
Cadeira,26593.039339
Notebook,26097.275945
Monitor,25552.246986
Cadeira Gamer,3000.000000
Mesa de Escritório,850.000000
Mouse sem fio,426.666667


<p>Dentro da região sudeste, Mouse é o produto preferido dos clientes e o produto menos requisitado é a Webcam.</p>

<h4>Análise de Taxas de Cancelamento:</h4>

<h5>Hipótese:</h5>
<h5>A categoria "Móveis" possui a maior taxa de pedidos cancelados ou devolvidos em comparação com "Eletrônicos" e "Informática"?</h5>

In [58]:
df_cancelado_devolvido = df[(df['Status_Pagamento'] == 'Cancelado') | (df['Status_Pagamento'] == 'Devolvido')]

In [59]:
taxa = (
    df_cancelado_devolvido.groupby('Categoria')['Status_Pagamento'].count() /
    df_cancelado_devolvido['Status_Pagamento'].count() * 100
).sort_values(ascending=False).to_frame()

taxa

,Status_Pagamento
Categoria,
Informática,72.222222
Móveis,16.666667
Eletrônicos,11.111111


<p>Após filtrar o dataset para os status <b>Cancelado</b> e <b>Devolvido</b>, calculei as taxas de pedidos <b>cancelados</b> ou <b>devolvidos</b> e agrupei por <b>categoria</b>.</p>
<p>A categoria <b>Informática</b> ficou com uma taxa de <b>72% de pedidos cancelados ou devolvidos</b>, já <b>Móveis</b> ficou com <b>16%</b> e <b>Eletrônicos</b>, com <b>11%</b>.</p>

<p>A partir disso, temos dados o suficiente para rejeitar essa hipótese. A categoria <b>Móveis</b> não possui a maior taxa de pedidos cancelados ou devolvidos se comparado a <b>Eletrônicos</b> e <b>Informática</b>.</p>

<p>Agora temos que identificar o porquê da categoria Informática ter uma taxa de pedidos cancelados ou devolvidos muito maior que as demais categorias. Será que a concorrência está ofertando produtos de informática num preço menor que o nosso? Será que os nossos produtos de informática estão vindo defeituosos da fábrica ou do fornecedor? Responder questionamentos como esses pode nos ajudar a traçar um plano de ação bem estruturado.</p>

<h4>Taxa de pedidos pendentes ou cancelados:</h4>

<h5>Hipótese:</h5>
<h5>Existe uma concentração maior de vendas com status "Pendente" ou "Cancelado" na segunda quinzena do mês em comparação com a primeira quinzena?</h5>

In [60]:
df['Quinzena'] = (
    df['Data'].dt.day.apply(
        lambda x: '1ª Quinzena' if x <= 15 else '2ª Quinzena'
    )
)

<p>Para realizar essa análise tive que criar uma nova coluna contendo as separações dos meses em quinzenas.</p>

In [61]:
df.head()

,ID_Venda,Data,Vendedor,Regiao,Produto,Categoria,Qtd,Preco_Unit,Total,Status_Pagamento,Quinzena
0,1,2026-01-15,João Silva,Sudeste,Notebook,Informática,2,3500.0,7000.0,Pago,1ª Quinzena
1,2,2026-01-16,Maria,Sul,Monitor,Eletrônicos,1,800.5,800.5,Pendente,2ª Quinzena
2,3,2026-01-18,João Silva,Sudeste,Notebook,Informática,1,3500.0,3500.0,Pago,2ª Quinzena
3,4,2026-01-20,Carlos Souza,Nordeste,Teclado Mecânico,Informática,5,150.0,750.0,Pago,2ª Quinzena
4,5,2026-01-22,Ana Lima,Não informado,Mouse sem fio,Informática,2,80.0,160.0,Cancelado,2ª Quinzena


In [62]:
df_pendente_cancelado = df[(df['Status_Pagamento'] == 'Pendente') | (df['Status_Pagamento'] == 'Cancelado')]
df_pendente_cancelado.head()

,ID_Venda,Data,Vendedor,Regiao,Produto,Categoria,Qtd,Preco_Unit,Total,Status_Pagamento,Quinzena
1,2,2026-01-16,Maria,Sul,Monitor,Eletrônicos,1,800.5,800.5,Pendente,2ª Quinzena
4,5,2026-01-22,Ana Lima,Não informado,Mouse sem fio,Informática,2,80.0,160.0,Cancelado,2ª Quinzena
14,14,2026-02-08,João Silva,Sudeste,Notebook,Informática,1,3500.0,3500.0,Pendente,1ª Quinzena
15,15,2026-02-10,Ana Lima,Sul,Webcam 1080p,Informática,3,250.0,750.0,Cancelado,1ª Quinzena
20,21,2026-02-22,Pedro Alves,Norte,Notebook,Informática,1,3500.0,3500.0,Cancelado,2ª Quinzena


<p>Filtrar o dataset para os status de pagamento <b>Pendente</b> e <b>Cancelado</b>.</p>

In [63]:
df_sumarizacao = df_pendente_cancelado.groupby(['Data', 'Quinzena'])['Status_Pagamento'].count().sort_values(ascending=False).reset_index()
df_sumarizacao

,Data,Quinzena,Status_Pagamento
0,2025-10-22,2ª Quinzena,7
1,2025-06-02,1ª Quinzena,7
2,2025-05-21,2ª Quinzena,6
3,2025-08-26,2ª Quinzena,5
4,2025-10-18,2ª Quinzena,5
...,...,...,...
430,2025-10-21,2ª Quinzena,1
431,2025-10-23,2ª Quinzena,1
432,2025-10-26,2ª Quinzena,1
433,2025-10-27,2ª Quinzena,1


<p>Agrupar por <b>Data</b> e <b>Quinzena</b> em cima do dataset filtrado, a contagem dos registros da coluna <b>Status_Pagamento</b>.</p>

In [64]:
df_sumarizacao[df_sumarizacao['Quinzena'] == '2ª Quinzena']['Status_Pagamento'].sum()

np.int64(404)

In [65]:
df_sumarizacao[df_sumarizacao['Quinzena'] == '1ª Quinzena']['Status_Pagamento'].sum()

np.int64(418)

<p>Por fim, verificamos que olhando para o macro, não mês a mês, rejeitamos a hipótese proposta.</p>
<p>A <b>concentração</b> maior de <b>vendas</b> com status <b>"Pendente"</b> ou <b>"Cancelado"</b> está na <b>primeira quinzena dos meses</b>, não na segunda quinzena, o que é contraintuitivo, pois os clientes costumam ter mais potencial de compra no início do mês.</p>

<h4>Desempenho de Produtos (Faturamento):</h4>

<h5>Hipótese:</h5>
<h5>O produto "Notebook" é responsável pela maior parcela do faturamento total, superando os demais produtos?</h5>

In [66]:
df.groupby('Produto')['Total'].sum().sort_values(ascending=False).to_frame()

,Total
Produto,
Headset,22617267.07
Teclado,22034908.28
Monitor,21763154.02
Cadeira,21469005.65
Mouse,20116547.19
Notebook,19859770.69
Cadeira Gamer,30000.00
Mesa de Escritório,17000.00
Webcam 1080p,5750.00


In [67]:
parcela_faturamento = (
    df.groupby('Produto')['Total'].sum() /
    df['Total'].sum() * 100
).sort_values(ascending=False).to_frame()

parcela_faturamento

,Total
Produto,
Headset,17.680295
Teclado,17.225056
Monitor,17.012621
Cadeira,16.782680
Mouse,15.725441
Notebook,15.524714
Cadeira Gamer,0.023452
Mesa de Escritório,0.013289
Webcam 1080p,0.004495


<p>O <b>produto</b> com o <b>maior percentual de faturamento</b> é o <b>Headset</b>, com quase <b>17,7%</b> do <b>faturamento total</b>.</p>
<p>O <b>Notebook</b> representa cerca de <b>15,5%</b> do <b>faturamento total</b>.</p>
<p>Logo, podemos dizer que o <b>Notebook não é responsável pela maior parcela do faturamento</b>.</p>

<h4>Desempenho dos Vendedores:</h4>

<h5>Hipótese:</h5>
<h5>Os vendedores apresentam diferenças significativas no faturamento médio por venda, indicando que alguns vendedores possuem maior desempenho comercial?</h5>

In [72]:
df_vendedor = df.groupby('Vendedor')['Total'].mean().sort_values(ascending=False).reset_index()
df_vendedor

,Vendedor,Total
0,Pedro Alves,26556.475130
1,João Silva,26544.265527
2,Carlos Souza,26297.038118
3,Ana Lima,26237.671378
4,Maria,26207.539597
5,Não identificado,700.250000


In [79]:
df_vendedor['Diferença'] = df_vendedor['Total'].diff().abs()
df_vendedor

,Vendedor,Total,Diferença
0,Pedro Alves,26556.475130,NaN
1,João Silva,26544.265527,12.209604
2,Carlos Souza,26297.038118,247.227409
3,Ana Lima,26237.671378,59.366740
4,Maria,26207.539597,30.131782
5,Não identificado,700.250000,25507.289597


<p>Após agrupar o faturamento médio por venda por vendedor, calculamos a diferença posicional das médias e constatamos que a diferença mais significativa é entre o vendedor João Silva e Carlos Souza, ou seja, João Silva vendeu em média aproximadamente R$247,22 a mais que Carlos Souza. </p>
<p>João Silva merece uma promoção!</p>
<p>As demais diferenças não são tão significativas.</p>

<h4>Resumo</h4>
<h5>Desempenho por Região (Ticket Médio):</h5> <p>Hipótese Validada</p>
<h5>Análise de Taxas de Cancelamento:</h5> <p>Hipótese Rejeitada</p>
<h5>Taxa de pedidos pendentes ou cancelados:</h5> <p>Hipótese Rejeitada</p>
<h5>Desempenho de Produtos (Faturamento):</h5> <p>Hipótese Rejeitada</p>
<h5>Desempenho dos Vendedores:</h5> <p>Hipótese Validada</p>